Step 1: Gather the "Patent Set"
    Source Dataset: SampleGloria_Pat_GlinerLabels.parquet
    Columns: patent_id, term
    Action: 1. Group by patent_id and term.
    2. Count the occurrences to get frequency in patent.
    3. This gives you your first "set" of terms for every patent.
    output -> patent_id, term, freq_in_patent

In [58]:
import pandas as pd

In [59]:
df = pd.read_parquet('SampleGloria_Pat_GlinerLabels.parquet')
df.head()

,patent_id,term,label,GrantedDate,year,num_claims,withdrawn
0,7662783,treating,Health Care Activity,2010-02-16,2010,9.0,0.0
1,7662783,solid tumor,Disease or Syndrome,2010-02-16,2010,9.0,0.0
2,7662783,patient,Human,2010-02-16,2010,9.0,0.0
3,7662783,comprising,Laboratory Procedure,2010-02-16,2010,9.0,0.0
4,7662783,administering,Health Care Activity,2010-02-16,2010,9.0,0.0


In [60]:
patent_terms = (
    df.groupby(["patent_id", "term"])
      .size()
      .reset_index(name="freq_in_patent")
)
patent_terms.freq_in_patent.unique()

array([1, 2, 4, 3, 6, 5])

Step 2: Bridge to the Papers
    Source Dataset: SampleGloria_Link_PmidOa.parquet
    Columns: patent_id, pmid
    Action: 1. Filter the pmid column to remove empty rows.
    2. Crucial: Extract the numeric ID from the end of the URL in the pmid column (e.g., from .../8623768 get 8623768).
    3. This tells you which Paper IDs belong to which Patent ID.
    output: "patent_id" and "pmid" (with just digit $ from the orginal string)

In [61]:
df = pd.read_parquet('SampleGloria_Link_PmidOa.parquet')
df.head(10)

,openalex,patent_id,oaid,scipat,confscore,ppp_score,pmid
0,W100031634,8145998,1.000316e+08,0.0,8.0,NaN,NaN
1,W100155009,7943636,1.001550e+08,0.0,10.0,NaN,https://pubmed.ncbi.nlm.nih.gov/8623768
2,W1002574413,7851144,1.002574e+09,0.0,4.0,NaN,https://pubmed.ncbi.nlm.nih.gov/3431463
3,W100376596,8008547,1.003766e+08,0.0,10.0,NaN,https://pubmed.ncbi.nlm.nih.gov/3273636
4,W10039931,8229677,1.003993e+07,0.0,10.0,NaN,https://pubmed.ncbi.nlm.nih.gov/11084856
5,W100681333,7662783,1.006813e+08,0.0,10.0,NaN,https://pubmed.ncbi.nlm.nih.gov/11191056
6,W1006955200,8178497,1.006955e+09,0.0,2.0,NaN,NaN
7,W1008795863,7741269,1.008796e+09,0.0,1.0,NaN,NaN
8,W101281971,8119608,1.012820e+08,0.0,10.0,NaN,https://pubmed.ncbi.nlm.nih.gov/11570495
9,W101336797,8222211,1.013368e+08,0.0,2.0,NaN,NaN


In [62]:
df_cleaned = df.dropna(subset= ["pmid"])
df_cleaned.head(30)
df_cleaned["pmid"] = df_cleaned["pmid"].str.extract(r'(\d+)$').astype(int) # split sullo / e -1
df_cleaned.head(20)

,openalex,patent_id,oaid,scipat,confscore,ppp_score,pmid
1,W100155009,7943636,1.001550e+08,0.0,10.0,NaN,8623768
2,W1002574413,7851144,1.002574e+09,0.0,4.0,NaN,3431463
3,W100376596,8008547,1.003766e+08,0.0,10.0,NaN,3273636
4,W10039931,8229677,1.003993e+07,0.0,10.0,NaN,11084856
5,W100681333,7662783,1.006813e+08,0.0,10.0,NaN,11191056
8,W101281971,8119608,1.012820e+08,0.0,10.0,NaN,11570495
12,W1021218198,8008547,1.021218e+09,0.0,10.0,NaN,2828857
13,W1021218198,8173393,1.021218e+09,0.0,10.0,NaN,2828857
16,W104312724,7785803,1.043127e+08,0.0,10.0,NaN,8774132
17,W105118458,7951356,1.051185e+08,0.0,10.0,NaN,3061932


Step 3: Gather the "Cited Papers Set"

    Source Dataset: SampleGloria_Pmed_GlinerLabels.parquet

    Columns: pmid, term

    Action: 1. Using the mapping from Step 2, link these terms back to their patent_id.
    2. Group by patent_id and term.
    3. Count the occurrences to get frequency in cited papers. (Note: If a patent cites 3 papers, this is the total count across all 3).

In [63]:
df_pmed = pd.read_parquet("SampleGloria_Pmed_GlinerLabels.parquet")
merged_df = pd.merge(df_pmed, df_cleaned, on=["pmid"], how="inner")
merged_df 
count_fre_in_pmid = (
    merged_df.groupby(["pmid", "term"])
      .size()
      .reset_index(name="freq_in_pmdi")
)
count_fre_in_pmid.head()


,pmid,term,freq_in_pmdi
0,2403803,administration,1
1,2403803,animal,1
2,2403803,antibody,1
3,2403803,antitumor,1
4,2403803,approach,1


Find the focal term

In [64]:
# Keep pmid in the groupby
cited_terms = (
    merged_df.groupby(["patent_id", "pmid", "term"])
             .size()
             .reset_index(name="freq_in_cited_paper")
)

In [65]:
cited_terms

,patent_id,pmid,term,freq_in_cited_paper
0,7662783,11387236,activity,1
1,7662783,11387236,adhesion,1
2,7662783,11387236,alpha,1
3,7662783,11387236,alpha5beta1,1
4,7662783,11387236,and,1
...,...,...,...,...
31287,8338460,12323058,reaction,1
31288,8338460,12323058,simplicity,1
31289,8338460,12323058,sulfur,1
31290,8338460,12323058,system,1


In [66]:
# Merge on patent_id + term → keeps only terms that appear in both
focal_terms = patent_terms.merge(cited_terms, on=["patent_id", "term"], how="inner")

In [67]:
# Save focal terms with pmid before aggregation
focal_terms_with_pmid = focal_terms[['patent_id', 'term', 'pmid']].rename(columns={'term': 'focal_term'})
focal_terms_with_pmid.to_parquet("output_dataset/focal_terms_with_pmid.parquet", index=False)
focal_terms_with_pmid.head()

,patent_id,focal_term,pmid
0,7662783,alpha,11387236
1,7662783,antagonist,11387236
2,7662783,cell,11535623
3,7662783,cell,12071856
4,7662783,cell,12454288


In [68]:
focal_terms

,patent_id,term,freq_in_patent,pmid,freq_in_cited_paper
0,7662783,alpha,1,11387236,1
1,7662783,antagonist,2,11387236,1
2,7662783,cell,1,11535623,1
3,7662783,cell,1,12071856,1
4,7662783,cell,1,12454288,1
...,...,...,...,...,...
1738,8309961,oxide,3,12764192,1
1739,8309961,second,1,12764192,1
1740,8309961,semiconductor,2,12764192,1
1741,8318804,cell,1,16557477,1


In [69]:
focal_terms = ( focal_terms.groupby(["patent_id", "term"], as_index=False) .agg( freq_in_patent=("freq_in_patent", "first"), freq_in_cited_paper=("freq_in_cited_paper", "sum"), ) )
focal_terms.rename(columns={'term': 'focal_term'}, inplace=True)


In [70]:
focal_terms

,patent_id,focal_term,freq_in_patent,freq_in_cited_paper
0,7662783,alpha,1,1
1,7662783,antagonist,2,1
2,7662783,cell,1,6
3,7662783,collagen,1,3
4,7662783,colon,1,1
...,...,...,...,...
785,8309961,oxide,3,1
786,8309961,second,1,1
787,8309961,semiconductor,2,1
788,8318804,cell,1,1


In [71]:
focal_terms.to_parquet("./output_dataset/focal_terms.parquet")